# 09 — Dong's linear Influence Model EM

We now choose one specific historical formulation: Dong's **linear marginal-state** algorithm. It propagates $CK$ marginal entries instead of $K^C$ joint states. For source chain $j$ and target chain $c$, assemble the block matrix

$$H_{(j,r),(c,s)}=D_{c,j}A_{c,j,r,s}.$$

The forward update is $\tilde\alpha_t=(\alpha_{t-1}H)\odot b_t$, followed by a separate normalization inside every target-chain block. This is a structured approximation to the full joint model; it is not the project-derived coordinate method from Tutorial 08.

## Expected counts become updates

The scaled backward pass produces $\gamma_{t,c,s}=\alpha_{t,c,s}\beta_{t,c,s}$. Its two-slice statistic assigns expected mass to every source/target transition block:

$$N_{c,j,r,s}=\sum_t \alpha_{t,j,r}H_{(j,r),(c,s)}\frac{b_{t+1,c,s}\beta_{t+1,c,s}}{q_{t+1,c}}.$$

Normalize $N$ over $s$ to update each $A_{c,j}$; sum each block and normalize over $j$ to update $D_{c,j}$. Emissions are ordinary $\gamma$-weighted symbol counts.

In [ ]:
import torch

from latentlinks.influence import InfluenceModel, dong_linear_em_step, dong_linear_inference

model = InfluenceModel(
    initial=torch.tensor([[0.6, 0.4], [0.5, 0.5]]),
    transition=torch.tensor([[[[0.9, 0.1], [0.2, 0.8]]] * 2] * 2),
    emission=torch.tensor([[[0.9, 0.1], [0.1, 0.9]]] * 2),
    influence=torch.tensor([[0.8, 0.2], [0.3, 0.7]]),
)
observations, _ = model.sample(40, seed=9)
posterior = dong_linear_inference(model, observations)
updated, before = dong_linear_em_step(model, [observations])
after = dong_linear_inference(updated, observations).log_likelihood
print("marginal shape:", tuple(posterior.marginals.shape))
print("linear objective:", float(before), "→", float(after))
assert after >= before - 1e-8

The quantity improved here is the linear algorithm's scaled marginal objective. Exact product-state inference remains the tiny-model oracle, and agreement is guaranteed in the independent-chain limit—not for arbitrary coupling.